*© 2026 Zettnq · oi-volume-signal-research · MIT License*
### 01. 1h Data Validation
Loading the raw 1-hour CSV, performing a full set of quality checks, and building a cleaned dataset. Artifact: data/processed/{asset}_{tf}_validated.parquet.

In [1]:
from pathlib import Path

import oivdc_research  # installed via `pip install -e .`

# repo root: .../repo/src/oivdcr_research/__init__.py -> parents[2] = repo
PROJECT_ROOT = Path(oivdc_research.__file__).resolve().parents[2]

import numpy as np
import pandas as pd

from oivdc_research.config import get_config

# Reproducibility parameters — edit here.
ASSET = "btcusdt"
TIMEFRAME = "1h"

CONFIG = get_config(
    project_root=PROJECT_ROOT,
    raw_csv_path=PROJECT_ROOT / "data" / "raw" / f"{ASSET}_{TIMEFRAME}.csv",
    timeframe=TIMEFRAME,
    asset=ASSET,
)

In [2]:
from oivdc_research.data import load_raw_data, validate_data

df_raw = load_raw_data(CONFIG)
df_checked, report = validate_data(df_raw, CONFIG)
report.show()

                     name  passed severity  n_issues                                                                                                            message examples
                 min_rows    True critical         0                                                                                Rows: 44502. Minimum required: 100.         
       timestamp_not_null    True critical         0                                                                                                No null timestamps.         
     timestamp_duplicates    True critical         0                                                                                           No duplicate timestamps.         
           numeric_no_nan    True critical         0                                                                                  No NaN values in numeric columns.       {}
           numeric_finite    True critical         0                                                               

In [3]:
from oivdc_research.data import prepare_validated_data, save_validated_data

assert report.critical_passed, "Critical validation checks failed — fix data first."

df_validated = prepare_validated_data(df_checked, CONFIG)
path = save_validated_data(df_validated, CONFIG)
print(f"Validated rows: {len(df_validated)} -> {CONFIG.display_path(path)}")

Validated rows: 44502 -> data/processed/btcusdt_1h_validated.parquet


Interpretation: critical checks (nulls, duplicates, prices, OHLC, finite) must pass; warning checks (frequency, return/delta_oi consistency) are documented in the article as limitations.

### 02. 5min Data Validation
Loading the raw 5-minute CSV, performing a full set of quality checks, and building a cleaned dataset. Artifact: data/processed/{asset}_{tf}_validated.parquet.

In [4]:
from oivdc_research.intrabar.data5m import load_5m, validate_5m, prepare_5m_data, save_5m_data

df5m_raw = load_5m(CONFIG)
df5m_checked, rep5m = validate_5m(df5m_raw, CONFIG, df1h=df_validated)
rep5m.show()

                name  passed severity  n_issues                                                   message examples
            min_rows    True critical         0                      Rows: 534024. Minimum required: 100.         
  timestamp_not_null    True critical         0                                       No null timestamps.         
timestamp_duplicates    True critical         0                                  No duplicate timestamps.         
      numeric_no_nan    True critical         0                                No NaN in numeric columns.         
      numeric_finite    True critical         0                                All numeric values finite.         
     prices_positive    True critical         0                                      All prices positive.         
  volume_nonnegative    True critical         0                                      volume non-negative.         
    ohlc_consistency    True critical         0                                 

In [5]:
assert rep5m.critical_passed, "Critical 5m checks failed."

df5m = prepare_5m_data(df5m_checked, CONFIG)
path5m = save_5m_data(df5m, CONFIG)
print(f"5m rows: {len(df5m)} -> {CONFIG.display_path(path5m)}")

5m rows: 534024 -> data/processed/btcusdt_5m.parquet
